# Project 1 - E-commerce

## Objectives

#### Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

#### Data Load

In [3]:
ecommerce_df = pd.read_csv("E-commerce.csv")

In [4]:
ecommerce_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


#### Data Understanding

This is a transactional level data of customer purchases from an online retail store

* InvoiceNo - A nominal 6-digit unique transaction number, c-cancellation

* StockCode (Product Item Code) -  A nominal 5-digit unique number for each distinct product

* Description - Product name

* Quantity - Quantities purchased per transaction

* InvoiceDate (Invoice Date Time) - The date and time of transaction 

* UnitPrice - Product price per unit in sterling

* CustomerID - A nominal 5-digit unique number for each customer

* Country - Country of the customer

#### Renaming columns for clarity

In [5]:
ecommerce_df.rename(columns={'StockCode':'ProductCode',
                             'Description': 'ProductName'}, inplace=True)

### Initial Data Exploration and Handling

In [6]:
ecommerce_df.shape

(541909, 8)

The dataset contains 5,41,909 rows and 8 columns

In [7]:
ecommerce_df.isnull().sum()

InvoiceNo           0
ProductCode         0
ProductName      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

#### Validating missing values of ProductName in CustomerID

In [8]:
ecommerce_df[
    ecommerce_df['ProductName'].isnull() &
    ecommerce_df['CustomerID'].notnull()
].shape

(0, 8)

In [9]:
ecommerce_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   ProductCode  541909 non-null  object 
 2   ProductName  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


`ISSUES`

1) InvoiceDate is stored as object
2) CustomerID has missing values
3) CustomerID is stored as float
4) ProductName has missing values

#### Converting InvoiceDate to DateTime

In [10]:
ecommerce_df['InvoiceDate'] = pd.to_datetime(ecommerce_df['InvoiceDate'])

#### Handling Missing CustomerID 

In [11]:
ecommerce_df['CustomerID'].isnull().sum()

np.int64(135080)

`OBSERVATION`

The CustomerID column contains 1,35,080 missing values.

#### Creating new dataframe

In [12]:
ecommerce_cleaned = ecommerce_df.dropna(subset=['CustomerID']).copy()

`ACTION`

Rows with missing CustomerID values were removed by creating a new cleaned dataset.

`REASON`

Since the objective of the project focuses on customer purchasing behavior,
a valid customer identifier is<br> essential for analysis.
Rows without CustomerID cannot be used for customer-level insights.

#### New dataframe shape

In [13]:
ecommerce_cleaned.shape

(406829, 8)

The new dataset contains 4,06,829 rows and 8 columns

#### CustomerID format validation

In [14]:
(ecommerce_cleaned['CustomerID'] % 1 != 0).sum()

np.int64(0)

There are zero CustomerID values with decimals other than .0

#### Converting CustomerID to INT

In [15]:
ecommerce_cleaned['CustomerID'] = ecommerce_cleaned['CustomerID'].astype(int)

#### Validating datatypes after conversion

In [16]:
ecommerce_cleaned.dtypes

InvoiceNo              object
ProductCode            object
ProductName            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
dtype: object

#### ProductName Handling

These rows were already removed during the handling of missing CustomerID. Hence, no separate handling needed.

## Exploratory Data Analysis (EDA)

#### Distribution of Quantity

In [48]:
ecommerce_cleaned['Quantity'].describe()

count    406829.000000
mean         12.061303
std         248.693370
min      -80995.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

`OBSERVATIONS`

* The Quantity contains both positive and negative values.
* The minimum quantity is -80995. Negative values might indicate product cancellations. This needs to be verified.
* The maximum quantity is 80995. Why is the maximum quantity so high?
* Most transactions fall between 2 to 12 items.
* The large difference between mean and median suggests a right skewed distribution with extreme outliers.
* Data is highly spread out due to some extreme values. The standard deviation is significantly higher than the mean.

#### Validating negative values

In [55]:
ecommerce_cleaned[ecommerce_cleaned['Quantity'] < 0][['InvoiceNo','Quantity']].head()

,InvoiceNo,Quantity
141,C536379,-1
154,C536383,-1
235,C536391,-12
236,C536391,-24
237,C536391,-24


`OBSERVATION`

Negative values does indicate product cancellations. Hence, verified.

#### Distribution of UnitPrice

In [44]:
ecommerce_cleaned['UnitPrice'].describe()

count    406829.000000
mean          3.460471
std          69.315162
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max       38970.000000
Name: UnitPrice, dtype: float64

`OBSERVATIONS`

* Most UnitPrices fall within a range of 1 to 4.
* Minimim price is 0. How can prices be 0? This needs to be verified.
* Maximum price is 38970. This needs to be verified.
* The difference between mean and median suggests a right skewed distribution. Few items are pulling the average up.
* Standard deviation is high compared to mean. This suggests that the data is highly spread out.

#### Unique customers

In [56]:
ecommerce_cleaned['CustomerID'].nunique()

4372

`OBSERVATION`

* The dataset contains 4,372 unique customers
* With a large number of transactions (4,06,829), having 4,372 unique customers means stronger participation